## Understanding the data

This section focuses on understanding what the data contains

In [2]:
import pandas as pd

def load_dataset():
    return pd.read_csv(
        "diabetes_binary_classification_data.csv"
    )

df = load_dataset()
df.head()

,Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0


The dataset contains the first 5 entries with their respective collums in which there are 22. Using the .info member function we can explore the different datatypes included inside the dataset. Additinally we use the .shape member function and see 253680 rows and 22 colums

In [4]:
df.info()
df.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 253680 entries, 0 to 253679
Data columns (total 22 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   Diabetes_binary       253680 non-null  float64
 1   HighBP                253680 non-null  float64
 2   HighChol              253680 non-null  float64
 3   CholCheck             253680 non-null  float64
 4   BMI                   253680 non-null  float64
 5   Smoker                253680 non-null  float64
 6   Stroke                253680 non-null  float64
 7   HeartDiseaseorAttack  253680 non-null  float64
 8   PhysActivity          253680 non-null  float64
 9   Fruits                253680 non-null  float64
 10  Veggies               253680 non-null  float64
 11  HvyAlcoholConsump     253680 non-null  float64
 12  AnyHealthcare         253680 non-null  float64
 13  NoDocbcCost           253680 non-null  float64
 14  GenHlth               253680 non-null  float64
 15  

(253680, 22)

Looking at the non-null count we see that there are no missing values, so handling of this isn't necessary.

The dataset only contains a range of datatypes such as, int64, float64 and objects. Looking further into what these objects contain by using the .value_counts function, reveal categorical data as text for MS zoning.

In [ ]:
df["MS Zoning"].value_counts()

In [ ]:
import matplotlib.pyplot as plt


df["SalePrice"].hist(bins=30)
plt.show()

## Cross examening features


Considering the 82 different features each house could have, we could search for data thats not valid across multple features, such as MS SubClass which contain both how many storyes the house contain and build period. Build year is availeble in another collum as is number of storyes.

Since 82 features is alot, developing a scoring system to determen which row may require additional attention enables (fullfør senere)


In [ ]:
df["MS SubClass"].value_counts()


In [ ]:

df["Year_Anomaly"] = (
    (df["MS SubClass"].isin([20, 60]) & (df["Year Built"] < 1946))
    |
    (df["MS SubClass"].isin([30, 70]) & (df["Year Built"] > 1945))
)

df["Style_Anomaly"] = (
    (df["MS SubClass"].isin([20, 30, 40, 120]) & (df["House Style"] != "1Story"))
    |
    (df["MS SubClass"].isin([50, 150]) & (df["House Style"] != "1.5Fin"))
    |
    (df["MS SubClass"].isin([60, 70, 160]) & (df["House Style"] != "2Story"))
    |
    (df["MS SubClass"].isin([80, 180]) & (df["House Style"] != "SLvl"))
    |
    (df["MS SubClass"].isin([85]) & (df["House Style"] != "SFoyer"))
)

df["Type_Anomaly"] = (
    (df["MS SubClass"] == 90) & (df["Bldg Type"] != "Duplex")
)
df["Year_Anomaly"] = (
    (df["MS SubClass"].isin([20, 60]) & (df["Year Built"] < 1946))
    |
    (df["MS SubClass"].isin([30, 70]) & (df["Year Built"] > 1945))
)

anomalies = df[
    df["Year_Anomaly"] |
    df["Style_Anomaly"] |
    df["Type_Anomaly"]
]


print("Year anomalies:", df["Year_Anomaly"].sum())
print("Style anomalies:", df["Style_Anomaly"].sum())
print("Type anomalies:", df["Type_Anomaly"].sum())


In [ ]:
df["Remodel_Anomaly"] = df["Year Remod/Add"] < df["Year Built"]
print("Remodel year anomalies:", df["Remodel_Anomaly"].sum())
df.loc[
    df["Remodel_Anomaly"],
    ["Year Built", "Year Remod/Add"]
]

Another peculiar anamoly would be having asbestos shinges as primary or secondary exterior past a certain time epoch. Since asbestos obsoletion depend on on many factors such as country regulations etc, a simple plot of both would uncover any anamolies, with the assumtion that any relativly newly built buildings with asbestos would question the integraty of the data.

In [ ]:


# Primary exterior = asbestos shingles
primary_asbestos = df[df["Exterior 1st"] == "AsbShng"]

plt.hist(primary_asbestos["Year Built"], bins=20, edgecolor="black")
plt.xlabel("Year Built")
plt.ylabel("Number of Houses")
plt.title("Asbestos Shingles as Primary Exterior")
plt.show()

In [ ]:
# Secondary exterior = asbestos shingles
secondary_asbestos = df[df["Exterior 2nd"] == "AsbShng"]

plt.hist(secondary_asbestos["Year Built"], bins=20, edgecolor="black")
plt.xlabel("Year Built")
plt.ylabel("Number of Houses")
plt.title("Asbestos Shingles as Secondary Exterior")
plt.show()

None of the plots seem to suggest any buildings built with asbestos past 1960 which makes sense considering it was banned around that time.

These are the categorizes we found that might be useful for predicting house prices. Using the corr() function to see if our assumption is correct.

In [ ]:
cols = ["Garage Area", "Lot Frontage", "Lot Area",
        "Overall Qual", "Overall Cond", "Year Built", "SalePrice"]

corr = df[cols].corr()
print(corr.round(2))

# Correlation with price only
print(corr["SalePrice"].drop("SalePrice").sort_values(ascending=False))

# Heatmap
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)

ax.set_xticks(range(len(cols)))
ax.set_yticks(range(len(cols)))
ax.set_xticklabels(cols, rotation=45, ha="right")
ax.set_yticklabels(cols)

# Annotate each cell
for i in range(len(cols)):
    for j in range(len(cols)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=9)

fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()